In [2]:
# import the necessary libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA

In [3]:
## import and read the dataset
data=pd.read_csv('credit_card.csv')
data.head()

,actionnr,merchant_id,average_amount_transaction_day,transaction_amount,is_declined,total_number_of_declines_day,isforeigntransaction,ishighriskcountry,daily_chargeback_avg_amt,sixmonth_avg_chbk_amt,sixmonth_chbk_freq,isfradulent
0,1,3160040998,100.0,3000.0,N,5,Y,Y,0,0.0,0,Y
1,2,3160040998,100.0,4300.0,N,5,Y,Y,0,0.0,0,Y
2,3,3160041896,185.5,4823.0,Y,5,N,N,0,0.0,0,Y
3,4,3160141996,185.5,5008.5,Y,8,N,N,0,0.0,0,Y
4,5,3160241992,500.0,26000.0,N,0,Y,Y,800,677.2,6,Y


In [4]:
## 

In [5]:
type(data)

pandas.core.frame.DataFrame

In [6]:
# Remove the target column from the dataset
data=data.drop(columns=['actionnr','merchant_id','isfradulent'])
data.head()

,average_amount_transaction_day,transaction_amount,is_declined,total_number_of_declines_day,isforeigntransaction,ishighriskcountry,daily_chargeback_avg_amt,sixmonth_avg_chbk_amt,sixmonth_chbk_freq
0,100.0,3000.0,N,5,Y,Y,0,0.0,0
1,100.0,4300.0,N,5,Y,Y,0,0.0,0
2,185.5,4823.0,Y,5,N,N,0,0.0,0
3,185.5,5008.5,Y,8,N,N,0,0.0,0
4,500.0,26000.0,N,0,Y,Y,800,677.2,6


In [7]:
## Identify the dataframe structure to  evaluvate meta data, assess the data quality, and understand of the dataset composition
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3075 entries, 0 to 3074
Data columns (total 9 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   average_amount_transaction_day  3075 non-null   float64
 1   transaction_amount              3075 non-null   float64
 2   is_declined                     3075 non-null   object 
 3   total_number_of_declines_day    3075 non-null   int64  
 4   isforeigntransaction            3075 non-null   object 
 5   ishighriskcountry               3075 non-null   object 
 6   daily_chargeback_avg_amt        3075 non-null   int64  
 7   sixmonth_avg_chbk_amt           3075 non-null   float64
 8   sixmonth_chbk_freq              3075 non-null   int64  
dtypes: float64(3), int64(3), object(3)
memory usage: 216.3+ KB


In [7]:
## Rename the columns of the dataframe

In [8]:
data = data.rename(columns={"isforeigntransaction": "is_foreign_transaction", "ishighriskcountry": "is_high_risk_country"})
print(data)

      average_amount_transaction_day  transaction_amount is_declined  \
0                              100.0              3000.0           N   
1                              100.0              4300.0           N   
2                              185.5              4823.0           Y   
3                              185.5              5008.5           Y   
4                              500.0             26000.0           N   
...                              ...                 ...         ...   
3070                           500.0             11000.0           Y   
3071                           800.0                 0.0           Y   
3072                           800.0             20800.0           Y   
3073                          1500.0             12000.0           Y   
3074                          1500.0             36000.0           Y   

      total_number_of_declines_day is_foreign_transaction  \
0                                5                      Y   
1            

In [9]:
## Converting the categorical attributes into  numerical values

In [9]:
encoder=OneHotEncoder(sparse_output=False,dtype=int)

In [11]:
encoded_data=encoder.fit_transform(data[['is_declined','is_foreign_transaction','is_high_risk_country']])
encoded_data

array([[1, 0, 0, 1, 0, 1],
       [1, 0, 0, 1, 0, 1],
       [0, 1, 1, 0, 1, 0],
       ...,
       [0, 1, 1, 0, 1, 0],
       [0, 1, 0, 1, 0, 1],
       [0, 1, 0, 1, 0, 1]])

In [36]:
encoded_df = pd.DataFrame(
    encoded_data,
)

print(encoded_df.head())

   0  1  2  3  4  5  6  7
0  1  0  0  1  0  1  0  1
1  1  0  0  1  0  1  0  1
2  0  1  1  0  1  0  0  1
3  0  1  1  0  1  0  0  1
4  1  0  0  1  0  1  0  1


In [12]:
cols = ['is_declined','is_foreign_transaction','is_high_risk_country']

encoder = OneHotEncoder(
    sparse_output=False,
    drop='if_binary',
    handle_unknown='ignore'
)

encoded_data = encoder.fit_transform(data[cols])

# Replace values in the same columns
data[cols] = encoded_data

data.head()

,average_amount_transaction_day,transaction_amount,is_declined,total_number_of_declines_day,is_foreign_transaction,is_high_risk_country,daily_chargeback_avg_amt,sixmonth_avg_chbk_amt,sixmonth_chbk_freq
0,100.0,3000.0,0.0,5,1.0,1.0,0,0.0,0
1,100.0,4300.0,0.0,5,1.0,1.0,0,0.0,0
2,185.5,4823.0,1.0,5,0.0,0.0,0,0.0,0
3,185.5,5008.5,1.0,8,0.0,0.0,0,0.0,0
4,500.0,26000.0,0.0,0,1.0,1.0,800,677.2,6


In [37]:
## combined the dataframes together
combined_cols = pd.concat([data,encoded_df], axis=1)
combined_cols.head()

,average_amount_transaction_day,transaction_amount,total_number_of_declines_day,daily_chargeback_avg_amt,sixmonth_avg_chbk_amt,sixmonth_chbk_freq,isfradulent,is_declined_N,is_declined_Y,is_foreign_transaction_N,...,isfradulent_N,isfradulent_Y,0,1,2,3,4,5,6,7
0,100.0,3000.0,5,0,0.0,0,Y,1,0,0,...,0,1,1,0,0,1,0,1,0,1
1,100.0,4300.0,5,0,0.0,0,Y,1,0,0,...,0,1,1,0,0,1,0,1,0,1
2,185.5,4823.0,5,0,0.0,0,Y,0,1,1,...,0,1,0,1,1,0,1,0,0,1
3,185.5,5008.5,8,0,0.0,0,Y,0,1,1,...,0,1,0,1,1,0,1,0,0,1
4,500.0,26000.0,0,800,677.2,6,Y,1,0,0,...,0,1,1,0,0,1,0,1,0,1


In [19]:
## Remove the categorical (original norminal) data columns from dataframe

In [34]:
data=combined_cols.drop(['is_declined','is_foreign_transaction','is_high_risk_country',], axis=1)
data.head()

,average_amount_transaction_day,transaction_amount,total_number_of_declines_day,daily_chargeback_avg_amt,sixmonth_avg_chbk_amt,sixmonth_chbk_freq,isfradulent,is_declined_N,is_declined_Y,is_foreign_transaction_N,is_foreign_transaction_Y,is_high_risk_country_N,is_high_risk_country_Y,isfradulent_N,isfradulent_Y
0,100.0,3000.0,5,0,0.0,0,Y,1,0,0,1,0,1,0,1
1,100.0,4300.0,5,0,0.0,0,Y,1,0,0,1,0,1,0,1
2,185.5,4823.0,5,0,0.0,0,Y,0,1,1,0,1,0,0,1
3,185.5,5008.5,8,0,0.0,0,Y,0,1,1,0,1,0,0,1
4,500.0,26000.0,0,800,677.2,6,Y,1,0,0,1,0,1,0,1


In [22]:
## check the correlational coefficient

In [35]:
data.corr()

ValueError: could not convert string to float: 'Y'